# RSNA Knee — pack the cache into shards

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-001** | `hlly34/rsna-knee-cache-pack` | EXP-001 cache re-packed into ~1 GB tar shards so it can be downloaded off Kaggle (vast.ai) in a few requests instead of 4,407 |

Kaggle's API downloads notebook outputs file by file, so 4,407 `.npz` files are impractical to fetch on a rented GPU. This notebook re-packs `rsna-knee-cache` into a few plain `.tar` shards (the npz files are already compressed) plus a manifest. `scripts/vast/get_data.sh` downloads and unpacks them.

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-001", "notebook": "hlly34/rsna-knee-cache-pack", "what": "EXP-001 cache re-packed into ~1 GB tar shards so it can be downloaded off Kaggle (vast.ai) in a few requests instead of 4,407"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=6):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


In [ ]:
import hashlib, tarfile
CACHE = walk_find(lambda d, f: d if os.path.basename(d) == "cache_256" and any(x.endswith(".npz") for x in f[:50]) else None)
files = sorted(f for f in os.listdir(CACHE) if f.endswith(".npz"))
total = sum(os.path.getsize(os.path.join(CACHE, f)) for f in files)
n_shards = max(1, round(total / 2**30))            # ~1 GiB per shard
manifest = {"studies": len(files), "bytes": total, "shards": []}
for k in range(n_shards):
    name = f"cache_256_part{k:02d}.tar"
    with tarfile.open(f"/kaggle/working/{name}", "w") as tar:
        for f in files[k::n_shards]:
            tar.add(os.path.join(CACHE, f), arcname=f"cache_256/{f}")
    h = hashlib.sha256(open(f"/kaggle/working/{name}", "rb").read()).hexdigest()
    manifest["shards"].append({"file": name, "sha256": h, "bytes": os.path.getsize(f"/kaggle/working/{name}")})
    print(name, manifest["shards"][-1]["bytes"] / 2**20, "MiB")
json.dump(manifest, open("/kaggle/working/cache_manifest.json", "w"), indent=1)
EXP.update(studies=len(files), size_gib=round(total / 2**30, 2), shards=n_shards)
json.dump(EXP, open("/kaggle/working/experiment.json", "w"), indent=1)
print(EXP)